In [6]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/perceived_speech/gpt_words_5/wheretheressmoke.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['story_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('wheretheressmoke', 'WER'): np.float64(7.155794176075754), ('wheretheressmoke', 'BLEU'): np.float64(5.9819780851328215), ('wheretheressmoke', 'METEOR'): np.float64(6.147434015058574), ('wheretheressmoke', 'BERT'): np.float32(14.002044)}


In [7]:
window_zscores = {'gpt_layer': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for gpt_layer in [3,4,5,8,10]:
    for task in ['wheretheressmoke']:
        scores = np.load(f'scores/S1/perceived_speech/gpt_words_{gpt_layer}/{task}.npz', allow_pickle=True)['window_zscores'].item()
        # print(scores['window_zscores'].item())
        window_zscores['gpt_layer'].append(gpt_layer)
        window_zscores['WER'].append(scores[(task, 'WER')])
        window_zscores['BLEU'].append(scores[(task, 'BLEU')])
        window_zscores['METEOR'].append(scores[(task, 'METEOR')])
        window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'gpt_layer': [3, 4, 5, 8, 10],
 'WER': [array([-5.37826588e-02,  8.87179157e-01,  7.01598259e-01,  5.60855047e-01,
          7.29414201e-01,  2.24651482e+00,  1.08236126e+00,  1.65503185e+00,
          2.08617312e+00,  1.84191450e+00,  1.99827230e+00,  2.43723456e+00,
          2.12277840e+00,  2.86556127e+00,  3.00057885e+00,  2.81019506e+00,
          2.24106129e+00,  2.77235530e+00,  1.41674271e+00,  7.65723925e-01,
          1.77561563e+00,  8.44763720e-01,  8.05242191e-01,  9.25237295e-01,
          9.27782833e-01,  1.53359590e-01,  2.01527509e-01,  6.16571666e-01,
          1.05099994e+00, -2.23662720e-02,  5.65735715e-01,  9.10142915e-01,
          9.23281351e-01,  4.25413563e-01,  1.93302542e-01,  9.03786479e-02,
          1.75852269e+00,  1.28533196e+00,  7.94882912e-01,  1.15550277e+00,
          5.94353290e-01,  9.11701011e-01,  6.21260595e-01, -6.87782156e-02,
          1.97620157e-01,  3.68855557e-02,  1.25390546e+00,  1.12419978e-01,
         -2.92875964e-01, -4.57471006

In [8]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

563
563
563
=
1689


,gpt_layer,WER,BLEU,METEOR,BERT
0,3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,4,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,5,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [4]:
results_df

,gpt_layer,WER,BLEU,METEOR,BERT
0,6,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,7,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,8,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,10,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."


In [9]:
gpt_layer_6=np.array(results_df.loc[0, 'BERT']).mean()
gpt_layer_7=np.array(results_df.loc[1, 'BERT']).mean()
gpt_layer_8=np.array(results_df.loc[2, 'BERT']).mean()
gpt_layer_9=np.array(results_df.loc[3, 'BERT']).mean()
gpt_layer_10=np.array(results_df.loc[4, 'BERT']).mean()
to_file = pd.DataFrame({'gpt_layer':[3,4,5,8,10], 'significantly_decoded': [gpt_layer_6,gpt_layer_7,gpt_layer_8,gpt_layer_9,gpt_layer_10]})
to_file.to_csv('perceived_speech_gpt_words.csv', index=False)

to_file

,gpt_layer,significantly_decoded
0,3,0.646536
1,4,0.658970
2,5,0.619893
3,8,0.648313
4,10,0.619893
